In [ ]:
from ppp_paths import databulk, dataint


In [ ]:
import pandas as pd
import unicodedata
import re

In [ ]:
possiblepairs = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')

In [ ]:
possiblepairs_patentsonly = possiblepairs['patent_id'].drop_duplicates().reset_index(drop=True)
possiblepairs_patentsonly
inventors = pd.read_csv(databulk + 'g_inventor_disambiguated.tsv.zip', sep='\t', compression='zip')
inventors = pd.merge(inventors, possiblepairs_patentsonly, on='patent_id', how='inner')
inventors = inventors[['patent_id', 'disambig_inventor_name_first', 'disambig_inventor_name_last']]
inventors.dropna(inplace=True)
inventors.drop_duplicates(inplace=True)
inventors['disambig_inventor_name_first'] = inventors['disambig_inventor_name_first'].str.replace(r'\ [A-Z]\.$','',regex=True).str.strip()
inventors['disambig_inventor_name_last'] = inventors['disambig_inventor_name_last'].str.replace(r'\,.*$','',regex=True).str.strip()
inventors['inventors'] = inventors['disambig_inventor_name_first'] + ' ' + inventors['disambig_inventor_name_last'] 
# inventors=inventors[['patent_id', 'inventorsoneline']].drop_duplicates()
inventors

In [ ]:
## list the suffixes to remove and nobiliary particles to merge

name_del = set(["2nd", "3rd", "Jr", "Jr.", "Junior", "Sr", "Sr.", "Senior", "III", "II"])
ln_suff= set(["oster", "nordre", "vaster", "aust", "vesle", "da", "van t", "af", "al", "setya", "zu", "la", "na", "mic", "ofver", "el", "vetle", "van het", "dos", "ui", "vest", "ab", "vste", "nord", "van der", "bin", "ibn", "war", "fitz", "alam", "di", "erch", "fetch", "nga", "ka", "soder", "lille", "upp", "ua", "te", "ni", "bint", "von und zu", "vast", "vestre", "over", "syd", "mac", "nin", "nic", "putri", "bet", "verch", "norr", "bath", "della", "van", "ben", "du", "stor", "das", "neder", "abu", "degli", "vre", "ait", "ny", "opp", "pour", "kil", "der", "oz",  "von", "at", "nedre", "van den", "setia", "ap", "gil", "myljom", "van de", "stre", "dele", "mck", "de", "mellom", "mhic", "binti", "ath", "binte", "snder", "sre", "ned", "ter", "bar", "le", "mala", "ost", "syndre", "sr", "bat", "sndre", "austre", "putra", "putera", "av", "lu", "vetch", "ver", "puteri", "mc", "tre", "st"])


#suppress all the unwanted suffixes from a string. 
#name_del file can be modified if more or less suffixes want to be suppressed 


def name_delete(string):
    for elmt in name_del:
        if f" {elmt}" in string:
            string =  string.replace(f" {elmt}","")
    return string

#normalize a string dat that represents often a name. 

def normalize(data):
    normal = unicodedata.normalize('NFKD', data).encode('ASCII', 'ignore')
    val = normal.decode("utf-8")
    # delete unwanted elmt
    val = name_delete(val)
    # Lower case
    val = val.lower()
    # remove special characters
    val = re.sub('[^A-Za-z ]+', ' ', val)
    # remove multiple spaces
    val = re.sub(' +', ' ', val)
    # remove trailing spaces
    val = val.strip()
    return val

normalize("PIERRE De BEHRMAN")



In [ ]:
if 'inventors' in inventors.columns:
    inventors['inventors'] = inventors['inventors'].apply(normalize)
    inventors['inventors'] = inventors['inventors']  + '#'
inventors

In [ ]:
inventors['inventors'] = inventors.groupby('patent_id')['inventors'].transform(lambda x: ''.join(x))
inventors.drop_duplicates(inplace=True)
inventors['inventors'] = inventors['inventors'].str.replace(r'(.*)#$', r'\1', regex=True)
inventors=inventors[['patent_id', 'inventors']].drop_duplicates()
inventors['patent_id'] = inventors['patent_id'].astype(str)
inventors

In [ ]:
def format_number(x):
    if pd.isna(x):
        return x  # Preserve NaN
    if isinstance(x, float) and x.is_integer():
        return str(int(x))  # Convert to int, then to string
    return str(x)  # Keep as is for non-integer
possiblepairs_papersonly = possiblepairs['magid'].drop_duplicates().reset_index(drop=True)
possiblepairs_papersonly
authors = pd.read_csv(databulk + "paperauthororder.zip", sep='\t', compression='zip')
authors = authors.rename(columns={'paperid': 'magid'})
authors['magid'] = authors['magid'].apply(format_number)
authors['authorid'] = authors['authorid'].astype(str)
authors = authors[['magid', 'authorid']]
authors.dropna(inplace=True)
authors = pd.merge(authors, possiblepairs_papersonly, on='magid', how='inner')
authors
authors.dropna()
author_names = pd.read_csv(databulk + "authoridname_normalized.zip", sep='\t', compression='zip')
author_names['authorid'] = author_names['authorid'].astype(str)
authors = pd.merge(authors, author_names, on=['authorid'], how='left')
authors.drop_duplicates(inplace=True)
authors.dropna(inplace=True)
authors = authors[['magid', 'authorname_normalized']]
authors.groupby('magid').head(30).reset_index()
authors['authorname_normalized'] = authors.groupby('magid')['authorname_normalized'].transform(lambda x: '#'.join(x))
authors = authors[['magid', 'authorname_normalized']].drop_duplicates()
authors['authorname_normalized'] = authors['authorname_normalized'].str.replace(r'(.*)#$', r'\1', regex=True)
authors = authors.rename(columns={'authorname_normalized': 'authors'})
authors = authors[['magid', 'authors']].drop_duplicates()
authors = authors.drop_duplicates().reset_index(drop=True)
authors

In [ ]:
possiblepairsinventors = pd.merge(possiblepairs, inventors, on='patent_id', how='left')
possiblepairsauthorsinventors = pd.merge(possiblepairsinventors, authors, on='magid', how='left')

In [ ]:
#remove middle initials from authors. could move this up
possiblepairsauthorsinventors['authors'] = possiblepairsauthorsinventors['authors'].str.replace(r' [a-z] ',' ',regex=True)
possiblepairsauthorsinventors['authors'] = possiblepairsauthorsinventors['authors'].str.replace(r' [a-z] ',' ',regex=True)
possiblepairsauthorsinventors['authors'] = possiblepairsauthorsinventors['authors'].str.replace(r' +',' ',regex=True)
possiblepairsauthorsinventors

In [ ]:
possiblepairsauthorsinventors.drop_duplicates(inplace=True)
possiblepairsauthorsinventors.dropna(inplace=True)
possiblepairsauthorsinventors  

In [ ]:
possiblepairsauthorsinventors.to_parquet(dataint + 'possiblepairsauthorsinventors.parquet', index=False, compression='snappy')